# WTI Oil Price Forecasting — One Agent, Three Tasks

> **Part 3 of 4.** This notebook builds on the agentic predictor introduced in
> [`02_intro_agentic_predictor.ipynb`](02_intro_agentic_predictor.ipynb).

A single Analyst Agent — backed by bounded Google Search — answers three tasks
using **one system prompt** and **task-specific user payloads**:

| Stream | Task | Output |
|--------|------|--------|
| A | Trajectory | 5/10/21-day price forecasts |
| B | Binary shock | P(WTI +$5 in 5 days) |
| C | Scenario analysis | Top 3 expert scenarios for 60 days |


In [ ]:
import json
import warnings

import pandas as pd


warnings.filterwarnings("ignore")

from aieng.forecasting.evaluation.task import ForecastingTask
from energy_oil_forecasting.analysis import compute_brier_score
from energy_oil_forecasting.data import WTI_SERIES_ID, build_wti_service, naive_utc_now
from energy_oil_forecasting.paths import (
    PROPHET_SHOCK_TRAJ_CACHE,
    PROPHET_TRAJ_CACHE,
    SCENARIO_CACHE,
    SCENARIO_ORIGIN,
    SHOCK_ANALYST_CACHE,
    SHOCK_HORIZON,
    SHOCK_ORIGINS,
    SHOCK_THRESHOLD,
    TRAJ_AGENT_CACHE,
    TRAJECTORY_ORIGINS,
)
from energy_oil_forecasting.prophet_baseline import (
    check_shock_outcome,
    load_prophet_trajectories,
    wti_series_to_price_df,
)
from energy_oil_forecasting.tasks import TASK_SPECS, build_wti_news_predictor
from energy_oil_forecasting.viz import prob_bar


data_service = build_wti_service()
ctx = data_service.context(as_of=naive_utc_now())
price_df = wti_series_to_price_df(ctx.get_series(WTI_SERIES_ID))

prophet_traj_df = load_prophet_trajectories(price_df, TRAJECTORY_ORIGINS, PROPHET_TRAJ_CACHE)
prophet_shock_df = load_prophet_trajectories(price_df, SHOCK_ORIGINS, PROPHET_SHOCK_TRAJ_CACHE)
print(f"Price history through {price_df.index[-1].date()}")

---
## Stream 1 — Trajectory Forecast

Compare Prophet fan charts to the news-grounded agent at three origins.

In [ ]:
trajectory_task = ForecastingTask(
    task_id="wti_trajectory_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[5, 10, 21],
    frequency="B",
    description="Trajectory demo for NB3",
)

traj_predictor = build_wti_news_predictor("trajectory")

if TRAJ_AGENT_CACHE.exists():
    with open(TRAJ_AGENT_CACHE) as f:
        traj_agent_results = json.load(f)
    print(f"Loaded {len(traj_agent_results)} cached trajectory agent runs.")
else:
    traj_agent_results = []
    for origin in TRAJECTORY_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = traj_predictor.predict(trajectory_task, origin_ctx)
        traj_agent_results.append(
            {
                "origin": str(origin.date()),
                "predictions": [p.model_dump(mode="json") for p in preds],
            }
        )
    with open(TRAJ_AGENT_CACHE, "w") as f:
        json.dump(traj_agent_results, f, indent=2)
    print(f"Saved {len(traj_agent_results)} agent trajectory runs.")

---
## Stream 2 — Binary Shock Prediction

In [ ]:
shock_task = ForecastingTask(
    task_id="wti_upshock_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[SHOCK_HORIZON],
    frequency="B",
    description="Binary upshock demo",
)

shock_predictor = build_wti_news_predictor("shock")

if SHOCK_ANALYST_CACHE.exists():
    with open(SHOCK_ANALYST_CACHE) as f:
        shock_results = json.load(f)
    print(f"Loaded {len(shock_results)} cached shock forecasts.")
else:
    shock_results = []
    for origin in SHOCK_ORIGINS:
        as_of = origin - pd.Timedelta(days=1)
        origin_ctx = data_service.context(as_of=as_of)
        preds = shock_predictor.predict(shock_task, origin_ctx)
        outcome, delta = check_shock_outcome(price_df, origin, SHOCK_THRESHOLD, SHOCK_HORIZON)
        shock_results.append(
            {
                "origin": str(origin.date()),
                "probability": preds[0].payload.probability,
                "outcome": outcome,
                "delta": delta,
                "metadata": preds[0].metadata,
            }
        )
    with open(SHOCK_ANALYST_CACHE, "w") as f:
        json.dump(shock_results, f, indent=2)

agent_probs = [r["probability"] for r in shock_results]
outcomes = [r["outcome"] for r in shock_results]
print(f"Agent Brier score: {compute_brier_score(agent_probs, outcomes):.4f}")
print(f"Task spec preview:\n{TASK_SPECS['shock'][:200]}...")

---
## Stream 3 — Scenario Analysis


In [ ]:
scenario_task = ForecastingTask(
    task_id="wti_scenario_demo",
    target_series_id=WTI_SERIES_ID,
    horizons=[21],
    frequency="B",
    description="Scenario analysis demo",
)

scenario_predictor = build_wti_news_predictor("scenario")

if SCENARIO_CACHE.exists():
    with open(SCENARIO_CACHE) as f:
        scenario_payload = json.load(f)
    print("Loaded cached scenario analysis.")
else:
    as_of = SCENARIO_ORIGIN - pd.Timedelta(days=1)
    origin_ctx = data_service.context(as_of=as_of)
    preds = scenario_predictor.predict(scenario_task, origin_ctx)
    scenario_payload = preds[0].metadata
    with open(SCENARIO_CACHE, "w") as f:
        json.dump(scenario_payload, f, indent=2)

for card in scenario_payload.get("scenarios", []):
    print(f"\n**{card['name']}**  P={card['probability']:.0%}  {prob_bar(card['probability'])}")
    print(card.get("description", ""))

---

## Summary

One agent identity (`build_wti_multitask_news_config` / `build_wti_news_config`) with
three task-specific prompt builders and output schemas demonstrates the bootcamp
pattern for multi-task agentic forecasting. Continue to
[`04_systematic_backtest_eval.ipynb`](04_systematic_backtest_eval.ipynb) for the
production backtest harness.
